# Task 1 Statistics Evidence - xfan0282


## Setup

This notebook shows the five Task 1 derived statistics for `xfan0282`.


In [ ]:
from importlib import import_module
import os
import pandas as pd

# PROJECT_ROOT and resolve_project_path keep file paths stable across local machines
from data2001.common.paths import PROJECT_ROOT, resolve_project_path
# load_settings reads configs/local.yaml and default paths
from data2001.config import load_settings
# run_task1_cleaning applies the group cleaning pipeline before individual statistics are calculated
from data2001.task1_cleaning.workflow import run_task1_cleaning
# extract_unit_column is the Task 1 cleaning step owned by xfan0282
from data2001.task1_cleaning.extract_unit_column import extract_unit_column



os.chdir(PROJECT_ROOT)

MEMBER_UNIKEY = "xfan0282"
settings = load_settings("configs/local.yaml") 
# load this member's statistic functions
statistics_module = import_module(f"data2001.task1_statistics.{MEMBER_UNIKEY}_statistics")

member_context = pd.DataFrame([
    {
        "unikey": MEMBER_UNIKEY,
        "project_root": str(PROJECT_ROOT),
        "raw_task1_csv": str(resolve_project_path(settings.outputs.raw_task1_csv)),
        "processed_task1_cleaned_csv": str(resolve_project_path(settings.outputs.processed_task1_cleaned_csv)),
    }
])
display(member_context)

## Shared Cleaning Input

This section creates the cleaned dataset used by `xfan0282` statistics.


### My Cleaning Step: Extracting Units from Description

My assigned cleaning step is `extract_unit_column()`. The raw Task 1 CSV stores the measurement unit inside the `Description` text, for example `Estimated resident population (no.)` or `Median age - persons (years)`. This is inconvenient for later statistics because the description mixes two meanings: the human-readable measure name and the unit used to interpret the number.

The implementation uses pandas string tools and a regular expression. It first copies the dataframe so the original input is not modified in place. Then it finds the description column, extracts the final bracketed text with `str.extract(r"\s*\(([^)]+)\)\s*$")`, removes that bracketed suffix from the description with `str.replace()`, and writes the extracted value into a new `unit` column. A few descriptions do not end with brackets, so the function also infers simple units from text patterns such as `Number of`, `Total number of`, `Census - Count of`, and descriptions starting with `%`.

This step matters because my Task 1 statistics compare percentages, counts, kilometres, years, and ratios. Separating `description` from `unit` makes the cleaned long dataset easier to filter, display, and explain without repeatedly parsing unit text later.


In [ ]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv) # raw CSV path used for this before/after cleaning demo

raw_preview = pd.read_csv(raw_task1_csv) # load the original CSV before the shared cleaning pipeline
unit_sample = raw_preview.loc[
    raw_preview["Description"].astype(str).str.contains(
        r"\(|^%|^Number of|^Total number of|^Census", regex=True, na=False
    ),
    ["Measure Code", "Description"],
].head(8) # select examples where the unit can be extracted or inferred

unit_sample_after = extract_unit_column(unit_sample) # run only xfan0282's cleaning step for before/after evidence

display(unit_sample.rename(columns={"Description": "description_before"})) # raw descriptions still contain units in brackets
display(
    unit_sample_after.rename(
        columns={"Description": "description_after", "unit": "unit_after"}
    )
) # cleaned descriptions are shorter and the extracted units are stored separately


The before table shows that the unit is embedded at the end of `Description`, so values such as counts, density, age, and percentages are only understandable after reading the full text. The after table keeps the measure name in `description_after` and moves the unit into `unit_after`, for example `no.`, `persons/km2`, and `years`. This makes the later statistics less ambiguous because the calculation can use clean measure codes and the explanation can use an explicit unit column.


In [ ]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv) # raw NSW statistics CSV from config
processed_task1_cleaned_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv) # cleaned CSV output path from config

cleaned_df = run_task1_cleaning(
    str(raw_task1_csv), # input CSV before shared cleaning
    str(processed_task1_cleaned_csv), # output CSV after shared cleaning
)

display(cleaned_df.head()) # preview cleaned columns and values after the group cleaning pipeline
display(pd.DataFrame([{"rows": len(cleaned_df), "columns": len(cleaned_df.columns)}])) # confirm cleaned dataset shape

## Individual Derived Statistics

This section runs the five `xfan0282` statistic functions and displays their returned `StatisticResult` rows.


In [ ]:
results = [] # stores successful StatisticResult dictionaries
errors = [] # stores function-level errors without stopping the whole notebook

for statistic_function in statistics_module.STATISTICS: # STATISTICS is the ordered function list in xfan0282_statistics.py
    try:
        result = statistic_function(cleaned_df) # each function receives the same shared cleaned dataframe
    except NotImplementedError:
        continue # skip placeholder functions if any remain
    except Exception as exc:
        errors.append({"function": statistic_function.__name__, "error": f"{type(exc).__name__}: {exc}"}) # keep visible evidence of failures
        continue
    results.append(result.to_dict()) # convert StatisticResult dataclass to displayable table rows

statistics_df = (
    pd.DataFrame(results) # pandas table for the five individual statistics
    .sort_values("statistic_id") # keep xfan0282-1 to xfan0282-5 in order
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 100) # keep descriptions readable in notebook output
display(
    statistics_df[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)

if errors:
    display(pd.DataFrame(errors)) # show any failed statistic function for debugging/evidence

## Explanation Notes

This section explains not only what the five derived statistics are, but also why each result led me to the next question. I wanted the statistics to form an exploration chain about how NSW changed between 2011, 2016, and 2021, rather than five unrelated numbers.

1. I started with dwelling structure because housing form is a basic signal of urban change. `xfan0282-1` shows that the apartment share increased by 2.90 percentage points from 2011 to 2021, reaching 21.72% in 2021. This is interesting because it suggests a shift toward higher-density living, which can affect demand for nearby services, transport, and public resources.

2. After seeing that residential structure was changing, I next checked whether daily work behaviour also changed. `xfan0282-2` shows that the work-from-home share grew 6.42 times from 2016 to 2021, rising from 4.82% to 30.98%. This is one of my strongest findings because it captures a large behavioural shift, likely connected to COVID-era work patterns, and it directly affects how people use local areas during the day.

3. The work-from-home result then led to a commuting question: if more people worked from home, did public transport commuting fall? `xfan0282-3` answers that question: the public transport commute share dropped by 11.98 percentage points from 2016 to 2021, falling from 15.98% to 4.00%. This is interesting because it supports the previous statistic with a related but different measure. Together, the two statistics tell a clearer story about reduced commuting demand rather than just one isolated change.

4. Once the overall commuting pattern was visible, I wanted to know whether commute burden was evenly distributed across workers. `xfan0282-4` shows a 5.30 km commute distance gap across occupation groups in 2016, with values ranging from 14.7 km to 20.0 km. This is interesting because averages can hide unequal travel burdens: some occupations may face longer travel distances even before the 2021 work-from-home shift.

5. Finally, I moved from mobility to affordability, because changes in housing and work are also connected to household pressure. `xfan0282-5` shows that rent stress in 2021 was 2.05 times mortgage stress: rent stress was 35.5%, while mortgage stress was 17.3%. This is interesting because it highlights that housing pressure is not evenly shared; renters appear to face much higher stress than mortgage holders.

The strongest report-ready findings are `xfan0282-2`, `xfan0282-3`, and `xfan0282-5`. They are interesting because they are easy to interpret, show large changes or contrasts, and connect to broader questions about post-COVID work patterns, transport demand, and housing affordability. `xfan0282-1` and `xfan0282-4` are still useful supporting findings because they provide context about density and unequal commuting burden.
